# Bottleneck analysis: HPC results

This notebook **only reads small result tables** from the HPC analysis script. It does not load checkpoints, image arrays, or feature caches, and does not import Torch or PennyLane. Run the processing on the HPC, copy back its exports, then Run all here.

The default experiment is `full128_array_12560`, comparing QUFEX with CNN replacement. The script performs branch interventions and linear probes without retraining the original networks.

## 1. Run the processing on HPC

From the repository root in a compute allocation, with the existing project environment:

```bash
uv run --no-sync python -u -m scripts.analyze_bottleneck \
  --run-id 12560 \
  --checkpoint-root /data/qmla/famato/checkpoints \
  --processed-cache /data/qmla/famato/data/processed/gz2-128-c5b8192136704e32 \
  --output-root /data/qmla/famato/results/bottleneck_analysis \
  --device cuda --batch-size 32 --cpu-threads 4
```

A submission template is provided in `scripts/slurm/bottleneck-analysis.sbatch.example`; edit its site-specific partition/account before using it. CPU execution is also supported with `--device cpu`.

The script reads the two `best.pt` checkpoints and all three processed splits. It writes feature arrays batch by batch and processes one model at a time. Fresh extraction is the default. `--reuse-features` reuses complete `.npy` caches produced by this script for the same inputs/settings; it does not reuse the previous notebook's `.npz` caches. Omit that flag after changing inputs or inference settings, or after incomplete extraction. No custom verification or cache freshness checks are performed.

## 2. Copy only the small results

Copy these **top-level files** from `/data/qmla/famato/results/bottleneck_analysis/full128_array_12560/` into local `results/bottleneck_analysis/full128_array_12560/`:

- `interventions.csv`
- `intervention_summary.csv`
- `probes.csv`
- `probe_selection.csv`
- `analysis_settings.json`
- Optionally, the two `.png` plots for viewing without Jupyter.

**Leave `features/`, the dataset, and the checkpoints on HPC.** This notebook regenerates plots from the CSVs. Restart your local kernel before using this version to release memory held by the earlier processing notebook. Change the path below if you copy results elsewhere.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "configs" / "experiments.toml").is_file()
)
pd.set_option("display.max_columns", 30)

In [ ]:
# EDIT THIS CELL.
RUN_ID = "12560"  # Also supports "12546" and "12554" after processing those runs.
RESULTS_ROOT = ROOT / "results" / "bottleneck_analysis"
EXPERIMENT = f"full128_array_{RUN_ID}"
OUTPUT_DIR = RESULTS_ROOT / EXPERIMENT
MODELS = ("qufex", "cnn_replacement")

settings = json.loads((OUTPUT_DIR / "analysis_settings.json").read_text(encoding="utf-8"))
interventions = pd.read_csv(OUTPUT_DIR / "interventions.csv")
intervention_summary = pd.read_csv(OUTPUT_DIR / "intervention_summary.csv")
probes = pd.read_csv(OUTPUT_DIR / "probes.csv")
probe_selection = pd.read_csv(OUTPUT_DIR / "probe_selection.csv")
print(f"Results: {OUTPUT_DIR}")
display(pd.DataFrame(settings["checkpoints"]).T)
print(f"HPC device: {settings['device']} | Batch size: {settings['batch_size']}")

## 1. Intervene on the trained bottleneck

Write the model as `head(z + b)`, where `z` is the compressed representation and `b` is the QUFEX or CNN branch. The head includes the post-bottleneck convolutions, global pooling, and classifier.

Compare original, removed, half-strength, and shuffled branches. Shuffling pairs each image with another position's branch output using a whole-test-set permutation. The same five seeds give the same permutations for both models because the shared dataset is read in manifest order. Permutations are not constrained to avoid occasional fixed points.

Everything stays in evaluation mode, including BatchNorm and dropout. No parameters are updated. A performance drop shows **dependence on this branch in the trained model**; it does not tell us whether the surrounding network could compensate after retraining.

In [ ]:
display(intervention_summary.round(4))

variants = ("removed", "half_strength", "shuffled")
x = np.arange(len(variants))
width = 0.36
fig, ax = plt.subplots(figsize=(8, 4))
for index, mode in enumerate(MODELS):
    frame = intervention_summary[intervention_summary["model"] == mode].set_index("variant").loc[list(variants)]
    ax.bar(
        x + (index - 0.5) * width,
        100 * frame["delta_macro_f1_mean"],
        width, label=mode,
        yerr=100 * frame["delta_macro_f1_std"], capsize=4,
    )
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x, ("Removed", "Half strength", "Shuffled"))
ax.set_ylabel("Macro-F1 change from original (percentage points)")
ax.set_title(f"{EXPERIMENT}: dependence on the bottleneck branch")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "intervention_macro_f1.png", dpi=180)
plt.show()

All deltas are **intervention minus original**: negative F1/accuracy deltas and positive cross-entropy deltas indicate worse performance. Cross-entropy is unweighted and uses natural logarithms.

Shuffled results are averaged over five permutations. Their standard deviation describes **permutation variability**, not uncertainty across independent training runs. Single-evaluation rows have zero plotted error bars.

A large removal/shuffling effect indicates that the head uses the branch. A small effect suggests limited dependence on this test set. Neither result alone proves that optimizing QUFEX's parameters was useful.

## 2. Measure class separability with linear probes

The HPC script uses cached, frozen features to fit three independent probes per model:

- **Compression:** flattened `z`.
- **Branch:** flattened `b`.
- **Residual sum:** flattened `z + b`.

Each fitted probe is a StandardScaler followed by logistic regression with balanced class weights. Fit both the scaler and classifier on **training data only**. Select regularization using **validation macro-F1**, then report test metrics once for the selected probe. No model selection uses test scores.

The four C values are considered in ascending order; an exact validation-score tie keeps the smaller C. Convergence warnings remain visible. Only these small classifiers are trained.

In [ ]:
display(probes.round(4))
display(probe_selection.round(4))

In [ ]:
representations = ("compression", "branch", "residual_sum")
x = np.arange(len(representations))
width = 0.36
fig, ax = plt.subplots(figsize=(8, 4))
for index, mode in enumerate(MODELS):
    frame = probes[probes["model"] == mode].set_index("representation").loc[list(representations)]
    ax.bar(x + (index - 0.5) * width, frame["macro_f1"], width, label=mode)
ax.set_xticks(x, ("Compression z", "Branch b", "Residual z + b"))
ax.set_ylabel("Test macro-F1")
ax.set_ylim(0, 1)
ax.set_title(f"{EXPERIMENT}: linear accessibility of class information")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "probe_macro_f1.png", dpi=180)
plt.show()

print("Residual-sum probe improvement over compression (percentage points):")
display(
    probes[probes["representation"] == "residual_sum"]
    .set_index("model")[["delta_macro_f1_vs_compression"]].mul(100).round(2)
)
print(f"Local tables, plots, and settings: {OUTPUT_DIR}")

## What to conclude

Read the intervention and probe results together:

- A branch-removal/shuffling penalty shows that the trained head depends on the branch.
- A better probe on `z + b` than on `z` indicates that the residual representation makes labels more accessible to a linear classifier.
- A strong branch-only probe shows that the branch carries decodable class information; it does not prove that the downstream head uses it.
- QUFEX and CNN replacement have separately trained compression networks. Compare the before/after changes within each model; their absolute scores do not isolate the bottleneck on identical inputs.

These are checkpoint-specific observations, not percentages of learning attributable to each component. They do not establish that **training the circuit parameters** caused the benefit; that requires a trained-versus-frozen-QUFEX experiment.

Changing the run ID allows the same analysis for `12546` or `12554`. Those runs also use seed 42 and differ in training settings, so do not treat them as independent-seed repetitions.